In [1]:
# 음성 데이터 로드 라이브러리
#!pip install librosa

In [1]:
import pandas as pd
import os
import numpy as np
import librosa
import time
from sklearn.model_selection import train_test_split

In [2]:
# 데이터프레임 불러오기
fourth_df = pd.read_csv(r"C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 음성 데이터셋\4차년도.csv", encoding='cp949')
fourth_df

,wav_id,발화문,상황,1번 감정,1번 감정세기,2번 감정,2번 감정세기,3번 감정,3번 감정세기,4번 감정,4번감정세기,5번 감정,5번 감정세기,나이,성별
0,5e258fd1305bcf3ad153a6a4,"어, 청소 니가 대신 해 줘!",anger,Neutral,0,Angry,1,Neutral,0,Neutral,0,Angry,1,27,male
1,5e258fe2305bcf3ad153a6a5,둘 다 청소 하기 싫어. 귀찮아.,anger,Neutral,0,Angry,1,Neutral,0,Neutral,0,Angry,1,27,male
2,5e258ff5305bcf3ad153a6a6,둘 다 하기 싫어서 화내.,anger,Angry,1,Angry,1,Neutral,0,Angry,1,Angry,1,27,male
3,5e25902f305bcf3ad153a6a9,그럼 방세는 어떡해.,anger,Sadness,1,Sadness,1,Sadness,1,Sadness,1,Sadness,1,27,male
4,5e27f90b5807b852d9e0157b,권태긴줄 알았는데 다른 사람이 생겼나보더라고.,sad,Sadness,1,Sadness,1,Sadness,1,Sadness,2,Sadness,1,32,male
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14601,5e53d6067bef803b4851dfc6,"아, 요즘 룸메랑 너무 자주 싸우게 돼.",anger,Disgust,1,Angry,2,Angry,1,Disgust,1,Angry,1,35,male
14602,5e53d61cc38c123b9ec6dde6,"아, 룸메가 방을 너무 지저분하게 써. 음식물도 막 버리고.",anger,Disgust,1,Angry,2,Angry,1,Disgust,1,Angry,1,35,male
14603,5e53d6332a2d173b73a03210,"뭐 화를 낸 것까진 아니지만, 한 달 전 쯤에 좀 확실하게 얘기를 해뒀거든. 근데 ...",anger,Disgust,1,Angry,2,Sadness,1,Disgust,1,Angry,2,35,male
14604,5e53d659963e443aee02b7d0,"어. 고등학교 동창인데, 같은 동네 오게 돼서 같이 룸메로 살게 됐지.",anger,Neutral,0,Angry,1,Sadness,1,Neutral,0,Neutral,0,35,male


In [3]:
print(fourth_df.columns)

Index(['wav_id', '발화문', '상황', '1번 감정', '1번 감정세기', '2번 감정', '2번 감정세기', '3번 감정',
       '3번 감정세기', '4번 감정', '4번감정세기', '5번 감정', '5번 감정세기', '나이', '성별'],
      dtype='object')


In [4]:
# 필요없는 칼럼 제거
drop_columns = ['1번 감정','1번 감정세기','2번 감정','2번 감정세기','3번 감정','3번 감정세기','4번 감정','4번감정세기','5번 감정','5번 감정세기']
fourth_df.drop(columns = drop_columns, axis=1, inplace=True)
fourth_df

,wav_id,발화문,상황,나이,성별
0,5e258fd1305bcf3ad153a6a4,"어, 청소 니가 대신 해 줘!",anger,27,male
1,5e258fe2305bcf3ad153a6a5,둘 다 청소 하기 싫어. 귀찮아.,anger,27,male
2,5e258ff5305bcf3ad153a6a6,둘 다 하기 싫어서 화내.,anger,27,male
3,5e25902f305bcf3ad153a6a9,그럼 방세는 어떡해.,anger,27,male
4,5e27f90b5807b852d9e0157b,권태긴줄 알았는데 다른 사람이 생겼나보더라고.,sad,32,male
...,...,...,...,...,...
14601,5e53d6067bef803b4851dfc6,"아, 요즘 룸메랑 너무 자주 싸우게 돼.",anger,35,male
14602,5e53d61cc38c123b9ec6dde6,"아, 룸메가 방을 너무 지저분하게 써. 음식물도 막 버리고.",anger,35,male
14603,5e53d6332a2d173b73a03210,"뭐 화를 낸 것까진 아니지만, 한 달 전 쯤에 좀 확실하게 얘기를 해뒀거든. 근데 ...",anger,35,male
14604,5e53d659963e443aee02b7d0,"어. 고등학교 동창인데, 같은 동네 오게 돼서 같이 룸메로 살게 됐지.",anger,35,male


In [5]:
# 데이터프레임에 음성파일 경로 추가
AUDIO_PATH = r"C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 음성 데이터셋\4차년도"
fourth_df['file_path'] = fourth_df['wav_id'].apply(lambda x: os.path.join(AUDIO_PATH, x + '.wav'))
fourth_df

,wav_id,발화문,상황,나이,성별,file_path
0,5e258fd1305bcf3ad153a6a4,"어, 청소 니가 대신 해 줘!",anger,27,male,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...
1,5e258fe2305bcf3ad153a6a5,둘 다 청소 하기 싫어. 귀찮아.,anger,27,male,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...
2,5e258ff5305bcf3ad153a6a6,둘 다 하기 싫어서 화내.,anger,27,male,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...
3,5e25902f305bcf3ad153a6a9,그럼 방세는 어떡해.,anger,27,male,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...
4,5e27f90b5807b852d9e0157b,권태긴줄 알았는데 다른 사람이 생겼나보더라고.,sad,32,male,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...
...,...,...,...,...,...,...
14601,5e53d6067bef803b4851dfc6,"아, 요즘 룸메랑 너무 자주 싸우게 돼.",anger,35,male,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...
14602,5e53d61cc38c123b9ec6dde6,"아, 룸메가 방을 너무 지저분하게 써. 음식물도 막 버리고.",anger,35,male,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...
14603,5e53d6332a2d173b73a03210,"뭐 화를 낸 것까진 아니지만, 한 달 전 쯤에 좀 확실하게 얘기를 해뒀거든. 근데 ...",anger,35,male,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...
14604,5e53d659963e443aee02b7d0,"어. 고등학교 동창인데, 같은 동네 오게 돼서 같이 룸메로 살게 됐지.",anger,35,male,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...


In [6]:
# csv와 음성파일 매칭시켜 음성 파일이 존재하지 않는 csv 제거
# 파일 존재여부 확인
def check_file_exists(file_path):
    if pd.isna(file_path): 
        return False
    return os.path.exists(file_path)

# file_exists column을 새로 만들어 file_path가 True인지 False인지 데이터프레임에 추가
fourth_df['file_exists'] = fourth_df['file_path'].apply(check_file_exists)

# file_exists가 True인 row만 선택해서 df_clean에 copy
df_clean = fourth_df[fourth_df['file_exists']].copy()
# file_exists column drop
df_clean.drop(columns=['file_exists'], inplace=True)

print(f"제거 전 갯수: {len(fourth_df)}")
print(f"제거 후 갯수: {len(df_clean)}")
print(f"제거된 갯수: {len(fourth_df) - len(df_clean)}")

제거 전 갯수: 14606
제거 후 갯수: 14590
제거된 갯수: 16


In [7]:
df_clean

,wav_id,발화문,상황,나이,성별,file_path
0,5e258fd1305bcf3ad153a6a4,"어, 청소 니가 대신 해 줘!",anger,27,male,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...
1,5e258fe2305bcf3ad153a6a5,둘 다 청소 하기 싫어. 귀찮아.,anger,27,male,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...
2,5e258ff5305bcf3ad153a6a6,둘 다 하기 싫어서 화내.,anger,27,male,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...
3,5e25902f305bcf3ad153a6a9,그럼 방세는 어떡해.,anger,27,male,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...
4,5e27f90b5807b852d9e0157b,권태긴줄 알았는데 다른 사람이 생겼나보더라고.,sad,32,male,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...
...,...,...,...,...,...,...
14601,5e53d6067bef803b4851dfc6,"아, 요즘 룸메랑 너무 자주 싸우게 돼.",anger,35,male,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...
14602,5e53d61cc38c123b9ec6dde6,"아, 룸메가 방을 너무 지저분하게 써. 음식물도 막 버리고.",anger,35,male,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...
14603,5e53d6332a2d173b73a03210,"뭐 화를 낸 것까진 아니지만, 한 달 전 쯤에 좀 확실하게 얘기를 해뒀거든. 근데 ...",anger,35,male,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...
14604,5e53d659963e443aee02b7d0,"어. 고등학교 동창인데, 같은 동네 오게 돼서 같이 룸메로 살게 됐지.",anger,35,male,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...


In [8]:
#상황 column을 감정으로 변경
df_clean.rename(columns={'상황':'감정'}, inplace=True)
#나이,성별 column drop => mfcc 중복 최소화
df_clean.drop(columns=['나이','성별'], inplace=True)
df_clean

,wav_id,발화문,감정,file_path
0,5e258fd1305bcf3ad153a6a4,"어, 청소 니가 대신 해 줘!",anger,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...
1,5e258fe2305bcf3ad153a6a5,둘 다 청소 하기 싫어. 귀찮아.,anger,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...
2,5e258ff5305bcf3ad153a6a6,둘 다 하기 싫어서 화내.,anger,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...
3,5e25902f305bcf3ad153a6a9,그럼 방세는 어떡해.,anger,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...
4,5e27f90b5807b852d9e0157b,권태긴줄 알았는데 다른 사람이 생겼나보더라고.,sad,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...
...,...,...,...,...
14601,5e53d6067bef803b4851dfc6,"아, 요즘 룸메랑 너무 자주 싸우게 돼.",anger,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...
14602,5e53d61cc38c123b9ec6dde6,"아, 룸메가 방을 너무 지저분하게 써. 음식물도 막 버리고.",anger,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...
14603,5e53d6332a2d173b73a03210,"뭐 화를 낸 것까진 아니지만, 한 달 전 쯤에 좀 확실하게 얘기를 해뒀거든. 근데 ...",anger,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...
14604,5e53d659963e443aee02b7d0,"어. 고등학교 동창인데, 같은 동네 오게 돼서 같이 룸메로 살게 됐지.",anger,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...


In [9]:
# MFCC : 음성 데이터를 특징 벡터화해주는 알고리즘
# file_path를 받아 특징 추출 함수 정의
def extract_features(file_path):
    try:
        # y : 음성데이터의 numpy 배열
        # sr :샘플링 속도 / sr=16000 : 초당 16000개의 샘플을 가지고 있는 데이터 / 16000인 이유 : 사람의 목소리는 대부분 16000Hz 안에 포함
        # liborsa : 음성 파일 불러오기
        y, sr = librosa.load(file_path, sr=16000) 
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return None
    # n_mfcc:mfcc의 개수를 정해주는 파라미터
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40) 
    mfccs_mean = np.mean(mfccs.T, axis=0) 
    
    return mfccs_mean

print("특징 추출")
start_time = time.time()

df_clean['features'] = df_clean['file_path'].apply(extract_features)

end_time = time.time()
print(f"--- 특징 추출 완료. 총 소요 시간 : {end_time - start_time:.2f}초 ---")

# 특징 추출 중 오류(None)가 발생한 행은 제거
df_clean.dropna(subset=['features'], inplace=True)

# 최종 확인
print("\n[추출 결과 확인]")
print(df_clean.head())
print(f"처리된 데이터 수: {len(df_clean)}개")
print(f"특징 벡터 크기: {df_clean['features'].iloc[0].shape if len(df_clean) > 0 else 'N/A'}")

특징 추출


C:\Users\user\anaconda3\Lib\site-packages\paramiko\pkey.py:82: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "cipher": algorithms.TripleDES,
C:\Users\user\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.Blowfish and will be removed from this module in 45.0.0.
  "class": algorithms.Blowfish,
C:\Users\user\anaconda3\Lib\site-packages\paramiko\transport.py:243: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "class": algorithms.TripleDES,


--- 특징 추출 완료. 총 소요 시간 : 679.94초 ---

[추출 결과 확인]
                     wav_id                        발화문     감정  \
0  5e258fd1305bcf3ad153a6a4           어, 청소 니가 대신 해 줘!  anger   
1  5e258fe2305bcf3ad153a6a5         둘 다 청소 하기 싫어. 귀찮아.  anger   
2  5e258ff5305bcf3ad153a6a6             둘 다 하기 싫어서 화내.  anger   
3  5e25902f305bcf3ad153a6a9                그럼 방세는 어떡해.  anger   
4  5e27f90b5807b852d9e0157b  권태긴줄 알았는데 다른 사람이 생겼나보더라고.    sad   

                                           file_path  \
0  C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...   
1  C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...   
2  C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...   
3  C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...   
4  C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...   

                                            features  
0  [-412.97424, 64.24774, 13.255594, 8.201703, -1...  
1  [-395.9492, 56.608116, 10.0962515, 0.8370305, ...  
2  [-407.18362, 47.00703, -0.15804905, 3.4194427,...  
3  [

In [19]:
df_clean

,wav_id,발화문,감정,file_path,features
0,5e258fd1305bcf3ad153a6a4,"어, 청소 니가 대신 해 줘!",anger,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...,"[-412.97424, 64.24774, 13.255594, 8.201703, -1..."
1,5e258fe2305bcf3ad153a6a5,둘 다 청소 하기 싫어. 귀찮아.,anger,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...,"[-395.9492, 56.608116, 10.0962515, 0.8370305, ..."
2,5e258ff5305bcf3ad153a6a6,둘 다 하기 싫어서 화내.,anger,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...,"[-407.18362, 47.00703, -0.15804905, 3.4194427,..."
3,5e25902f305bcf3ad153a6a9,그럼 방세는 어떡해.,anger,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...,"[-421.00314, 59.054688, 4.4928565, 8.804134, 1..."
4,5e27f90b5807b852d9e0157b,권태긴줄 알았는데 다른 사람이 생겼나보더라고.,sad,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...,"[-482.86273, 113.79381, 47.76626, 28.533253, 1..."
...,...,...,...,...,...
14601,5e53d6067bef803b4851dfc6,"아, 요즘 룸메랑 너무 자주 싸우게 돼.",anger,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...,"[-284.02798, 71.496574, 23.015589, 7.409676, -..."
14602,5e53d61cc38c123b9ec6dde6,"아, 룸메가 방을 너무 지저분하게 써. 음식물도 막 버리고.",anger,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...,"[-279.42603, 91.13584, 38.436283, 2.8053346, -..."
14603,5e53d6332a2d173b73a03210,"뭐 화를 낸 것까진 아니지만, 한 달 전 쯤에 좀 확실하게 얘기를 해뒀거든. 근데 ...",anger,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...,"[-290.65198, 76.50679, 23.249273, 9.698766, -4..."
14604,5e53d659963e443aee02b7d0,"어. 고등학교 동창인데, 같은 동네 오게 돼서 같이 룸메로 살게 됐지.",anger,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...,"[-328.26282, 76.848885, 29.647923, 13.668423, ..."


In [28]:
#추출된 40개의 음성 특징을 column으로 분리
features_df = pd.DataFrame(df_clean['features'].tolist(), index=df_clean.index)
# 기존의 데이터프레임과 합치기
df_final_csv = pd.concat([df_clean.drop('features', axis=1), features_df], axis=1)
# column이름 정리
new_columns = {i: f'feature_{i}' for i in range(features_df.shape[1])}
df_final_csv.rename(columns=new_columns, inplace=True)
#csv파일로 새로 저장하기
output_file_path = "preprocessing_fourth_df.csv"
df_final_csv.to_csv(output_file_path, index=False) 

In [26]:
df_final_csv

,wav_id,발화문,감정,file_path,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,...,feature_30,feature_31,feature_32,feature_33,feature_34,feature_35,feature_36,feature_37,feature_38,feature_39
0,5e258fd1305bcf3ad153a6a4,"어, 청소 니가 대신 해 줘!",anger,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...,-412.974243,64.247742,13.255594,8.201703,-1.359636,1.026765,...,-3.399848,0.156231,7.809394,9.621494,10.476119,14.216407,5.218826,8.790462,5.851394,1.509703
1,5e258fe2305bcf3ad153a6a5,둘 다 청소 하기 싫어. 귀찮아.,anger,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...,-395.949188,56.608116,10.096251,0.837030,-1.212133,3.972554,...,-6.465803,-1.485719,-2.689845,1.956669,1.832623,12.925749,3.906231,12.040658,9.189884,6.067553
2,5e258ff5305bcf3ad153a6a6,둘 다 하기 싫어서 화내.,anger,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...,-407.183624,47.007030,-0.158049,3.419443,10.055807,4.312028,...,-5.841687,1.226680,-1.121284,1.073898,2.544075,13.487154,4.974171,15.196485,7.993507,7.468258
3,5e25902f305bcf3ad153a6a9,그럼 방세는 어떡해.,anger,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...,-421.003143,59.054688,4.492857,8.804134,19.737129,-2.956160,...,-2.155815,0.251875,-3.269133,4.451177,1.988187,8.143342,2.350438,8.832993,1.775095,7.868490
4,5e27f90b5807b852d9e0157b,권태긴줄 알았는데 다른 사람이 생겼나보더라고.,sad,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...,-482.862732,113.793808,47.766258,28.533253,10.650901,6.818515,...,-2.482007,-1.061704,-0.654995,-0.289433,-0.741502,-1.598996,-1.474551,-0.601745,-1.115405,-0.715617
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14601,5e53d6067bef803b4851dfc6,"아, 요즘 룸메랑 너무 자주 싸우게 돼.",anger,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...,-284.027985,71.496574,23.015589,7.409676,-4.550021,15.085970,...,-6.243781,0.290705,-5.716758,-0.260983,-1.763641,1.786054,-3.568542,-1.741548,-4.750730,-3.403370
14602,5e53d61cc38c123b9ec6dde6,"아, 룸메가 방을 너무 지저분하게 써. 음식물도 막 버리고.",anger,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...,-279.426025,91.135841,38.436283,2.805335,-15.725999,20.172813,...,-4.784002,-0.014770,-4.217318,0.174063,-5.158162,-0.215629,-5.525555,-0.806506,-6.093986,-3.584964
14603,5e53d6332a2d173b73a03210,"뭐 화를 낸 것까진 아니지만, 한 달 전 쯤에 좀 확실하게 얘기를 해뒀거든. 근데 ...",anger,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...,-290.651978,76.506790,23.249273,9.698766,-4.611273,13.879671,...,-4.326666,-1.767662,-3.705427,-0.349082,-1.172038,0.846112,-1.744976,0.934780,-4.482302,-0.377516
14604,5e53d659963e443aee02b7d0,"어. 고등학교 동창인데, 같은 동네 오게 돼서 같이 룸메로 살게 됐지.",anger,C:\Users\user\OneDrive\바탕 화면\말동이\감정 분류를 위한 대화 ...,-328.262817,76.848885,29.647923,13.668423,-6.353665,9.780107,...,-4.505015,-0.888425,-3.924709,-1.221299,-2.391923,-1.475634,-2.682798,-1.045449,-4.275680,-1.377130
